# energy-anatomy: Kaggle T4 profiling run

Thin runner for `anatomy profile` (DESIGN section 6, docs/methods.md section 6). All logic lives in the repo; this notebook only clones, installs, checks the GPU, and runs.

**Notebook settings:** Accelerator = GPU T4 x2 (only device 0 is used), Internet = on, Persistence = off.

**Resuming across sessions:** Kaggle sessions end after 12 h. To continue a run, add the previous version of this notebook's output as an input (Add input > Notebook output). Cell 3 finds the latest `passes.csv`/`idle.csv` under `/kaggle/input` and copies them into the output folder before `--resume`.

**Smoke test (first session):** set `SMOKE = True`. It runs one 160M config into `/kaggle/working/smoke/`, separate from the real run.

In [ ]:
REPO = "https://github.com/scienseize/energy-anatomy"
REF = "main"          # branch or commit; the commit actually used is recorded per row in git_sha
SMOKE = False         # True: first-session smoke test only
OUT = "/kaggle/working/smoke" if SMOKE else "/kaggle/working/raw"
LIMIT = "--max-configs 1" if SMOKE else ""


In [ ]:
!git clone --quiet {REPO} /kaggle/working/energy-anatomy
!cd /kaggle/working/energy-anatomy && git checkout --quiet {REF} && git log -1 --oneline
!pip install --quiet -e "/kaggle/working/energy-anatomy[gpu]"
!nvidia-smi --query-gpu=index,name,pci.bus_id,driver_version,power.limit,clocks.max.sm --format=csv

In [ ]:
# Carry over passes from an earlier session (resume). Picks the file with the most rows.
import glob, os, shutil
os.makedirs(OUT, exist_ok=True)
for name in ("passes.csv", "idle.csv"):
    found = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True), key=lambda p: sum(1 for _ in open(p)))
    dest = os.path.join(OUT, name)
    if found and not os.path.exists(dest):
        shutil.copy(found[-1], dest)
        print(f"resumed {name} from {found[-1]} ({sum(1 for _ in open(dest)) - 1} rows)")
    elif not found:
        print(f"no previous {name}: fresh run")

In [ ]:
# Zeus must read nonzero energy on the T4 before spending GPU hours.
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
import torch
from zeus.monitor import ZeusMonitor
m = ZeusMonitor(gpu_indices=[0])
x = torch.randn(4096, 4096, device="cuda")
m.begin_window("check")
for _ in range(200):
    x = x @ x
    x = x / x.norm()
r = m.end_window("check")
e = sum(r.gpu_energy.values())
print(f"{torch.cuda.get_device_name(0)}: {e:.1f} J in {r.time:.2f} s ({e / r.time:.1f} W)")
assert e > 0, "Zeus read zero energy: stop and check NVML support"
del x; torch.cuda.empty_cache()

In [ ]:
# Provenance: which ATen ops F.rms_norm (unit rmsnorm_fused) dispatches to on this GPU and torch build.
# A single _fused_rms_norm op (and its backward) means the fused kernel ran; pow/mean/rsqrt/mul means it decomposed.
import torch, torch.nn.functional as F
from torch.utils._python_dispatch import TorchDispatchMode
class Ops(TorchDispatchMode):
    def __init__(self): super().__init__(); self.ops = []
    def __torch_dispatch__(self, func, types, args=(), kwargs=None):
        if not func.is_view: self.ops.append(str(func.overloadpacket))
        return func(*args, **(kwargs or {}))
for dt in (torch.float32, torch.float16):
    x = torch.randn(8, 512, 768, device="cuda", dtype=dt, requires_grad=True)
    w = torch.ones(768, device="cuda", dtype=dt, requires_grad=True)
    with Ops() as o:
        y = F.rms_norm(x, (768,), w, 1e-5); y.backward(torch.ones_like(y))
    print(torch.__version__, dt, "->", o.ops)

In [ ]:
# Provenance: does torch.compile fuse rope_fused on this GPU? Lists the CUDA kernels one fwd+bwd launches.
# Triton kernels (names starting "triton_") mean Inductor fused it; eager kernels mean it fell back.
import torch
from torch.profiler import profile, ProfilerActivity
from anatomy.config import load_model
from anatomy.substitutions import RopeFused
m = load_model("pythia-160m")
for dt in (torch.float32, torch.float16):
    mod = RopeFused(m, 512).to(device="cuda", dtype=dt)
    q, k = (torch.randn(8, m.heads, 512, m.head_dim, device="cuda", dtype=dt, requires_grad=True) for _ in range(2))
    for _ in range(3):  # compile + warm up
        a, b = mod(q, k); (a.sum() + b.sum()).backward()
    with profile(activities=[ProfilerActivity.CUDA]) as prof:
        a, b = mod(q, k); (a.sum() + b.sum()).backward(); torch.cuda.synchronize()
    names = sorted({e.key for e in prof.key_averages()})
    print(torch.__version__, dt, f"{sum(n.startswith('triton') for n in names)} triton kernels of {len(names)}:", names)

In [ ]:
!cd /kaggle/working/energy-anatomy && anatomy profile --resume --out {OUT} {LIMIT}

In [ ]:
# Progress summary: rows per status and per config.
import pandas as pd
p = pd.read_csv(f"{OUT}/passes.csv")
print(p.status.value_counts().to_string(), "\n")
print(p.groupby(["model", "seq_len", "precision"]).size().rename("rows").to_string())
print("\nthrottled passes:", int(p.throttled.fillna(False).astype(bool).sum()))
print("\nthrottle reasons:", p.throttle_reasons.value_counts().to_dict())
print("\nsessions (run_id -> GPU):")
print(p.groupby(["run_id", "gpu_uuid"]).size().rename("rows").to_string())